# Notebook 3: ESM2 + LoRA Fine-Tuning

Level 2 approach: fine-tune ESM2 with LoRA adapters for multi-label GO prediction.

Pipeline:
1. Load preprocessed CAFA6 splits
2. Build ESM2 + LoRA classifier
3. Train with BCE loss and cosine LR schedule
4. Evaluate with Fmax, AUPR per ontology
5. Push to HuggingFace Hub

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import torch
import yaml
import pickle
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm
from transformers import AutoTokenizer, get_cosine_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import DataLoader

from models.esm2_classifier import build_esm2_classifier, ESM2Classifier
from dataset import ESM2Dataset, collate_fn_pad
from evaluate import compute_fmax, compute_aupr, compute_coverage
from preprocessing import load_processed

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Configuration

In [ ]:
with open('../config/esm2_lora.yaml') as f:
    cfg = yaml.safe_load(f)

# Override for notebook experimentation
cfg['model']['base_model'] = 'facebook/esm2_t6_8M_UR50D'  # small model for quick testing
cfg['training']['num_epochs'] = 5
cfg['training']['batch_size'] = 4

ONTOLOGY = 'BPO'
DATA_PATH = f'../data/processed/cafa6_{ONTOLOGY.lower()}.pkl'
OUTPUT_DIR = f'../outputs/esm2_lora_{ONTOLOGY.lower()}'
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

## 2. Load Data

In [ ]:
data = load_processed(DATA_PATH)
go_terms = data['go_terms']
num_labels = len(go_terms)
print(f'Ontology: {ONTOLOGY} | GO terms: {num_labels:,}')

tokenizer = AutoTokenizer.from_pretrained(cfg['model']['base_model'])
max_len = cfg['model']['max_length']

def make_loader(split_key, shuffle):
    split = data[split_key]
    ds = ESM2Dataset(split['sequences'], tokenizer, max_len, split['labels'], split['protein_ids'])
    return DataLoader(ds, batch_size=cfg['training']['batch_size'],
                      shuffle=shuffle, collate_fn=collate_fn_pad, num_workers=0)

train_loader = make_loader('train', shuffle=True)
val_loader   = make_loader('val',   shuffle=False)
test_loader  = make_loader('test',  shuffle=False)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 3. Build Model

In [ ]:
model = build_esm2_classifier(cfg, num_labels, go_terms).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 4. Training Loop

In [ ]:
import torch.nn as nn

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=cfg['training']['learning_rate'],
    weight_decay=cfg['training']['weight_decay'],
)

num_epochs = cfg['training']['num_epochs']
grad_accum = cfg['training']['gradient_accumulation_steps']
total_steps = (len(train_loader) // grad_accum) * num_epochs
warmup_steps = int(total_steps * cfg['training']['warmup_ratio'])

scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
use_fp16 = cfg['training'].get('fp16', False) and torch.cuda.is_available()
scaler = torch.cuda.amp.GradScaler() if use_fp16 else None

history = {'train_loss': [], 'val_loss': [], 'fmax': [], 'aupr': []}

@torch.no_grad()
def evaluate_loader(loader):
    model.eval()
    logits_list, labels_list = [], []
    total_loss = 0
    for batch in loader:
        ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labs = batch['labels'].to(DEVICE)
        out = model(ids, mask, labs)
        total_loss += out['loss'].item()
        logits_list.append(out['logits'].cpu().numpy())
        labels_list.append(labs.cpu().numpy())
    scores = torch.sigmoid(torch.tensor(np.concatenate(logits_list))).numpy()
    y_true = np.concatenate(labels_list)
    return y_true, scores, total_loss / len(loader)

best_fmax = 0.0
for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_loss = 0.0
    optimizer.zero_grad()
    
    for step, batch in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}')):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        labs = batch['labels'].to(DEVICE)
        
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_fp16):
            out = model(ids, mask, labs)
            loss = out['loss'] / grad_accum
        
        (scaler.scale(loss) if scaler else loss).backward()
        epoch_loss += loss.item() * grad_accum
        
        if (step + 1) % grad_accum == 0:
            if scaler:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
    
    y_true, scores, val_loss = evaluate_loader(val_loader)
    fmax, best_t = compute_fmax(y_true, scores)
    aupr = compute_aupr(y_true, scores)
    coverage = compute_coverage(scores, best_t)
    
    train_loss_avg = epoch_loss / len(train_loader)
    history['train_loss'].append(train_loss_avg)
    history['val_loss'].append(val_loss)
    history['fmax'].append(fmax)
    history['aupr'].append(aupr)
    
    print(f'Epoch {epoch}/{num_epochs} | train_loss={train_loss_avg:.4f} | '
          f'val_loss={val_loss:.4f} | Fmax={fmax:.4f} (t={best_t:.2f}) | '
          f'AUPR={aupr:.4f} | Coverage={coverage:.3f}')
    
    if fmax > best_fmax:
        best_fmax = fmax
        model.save_pretrained(f'{OUTPUT_DIR}/best')
        print(f'  >>> New best Fmax={fmax:.4f} saved')

## 5. Training Curves

In [ ]:
epochs = list(range(1, num_epochs + 1))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, history['train_loss'], label='Train', marker='o')
axes[0].plot(epochs, history['val_loss'],   label='Val',   marker='s')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='BCE Loss')
axes[0].legend()

axes[1].plot(epochs, history['fmax'], color='green', marker='o')
axes[1].set(xlabel='Epoch', ylabel='Fmax', title='Validation Fmax')

axes[2].plot(epochs, history['aupr'], color='purple', marker='o')
axes[2].set(xlabel='Epoch', ylabel='AUPR', title='Validation AUPR')

plt.tight_layout()
plt.savefig(f'../reports/esm2_lora_{ONTOLOGY.lower()}_training.png', dpi=150)
plt.show()

## 6. Test Set Evaluation

In [ ]:
# Load best checkpoint
best_model = ESM2Classifier.from_pretrained(f'{OUTPUT_DIR}/best').to(DEVICE)
y_true_test, scores_test, _ = evaluate_loader(test_loader)

fmax_test, best_t_test = compute_fmax(y_true_test, scores_test)
aupr_test = compute_aupr(y_true_test, scores_test)
cov_test = compute_coverage(scores_test, best_t_test)

print(f'=== Test Set Results ({ONTOLOGY}) ===')
print(f'Fmax     : {fmax_test:.4f} (threshold={best_t_test:.2f})')
print(f'AUPR     : {aupr_test:.4f}')
print(f'Coverage : {cov_test:.4f}')

## 7. Push to HuggingFace Hub

In [ ]:
HUB_REPO_ID = 'your-username/cafa6-esm2-lora-bpo'  # change to your HF username
HF_TOKEN = 'your_hf_token_here'  # or use: huggingface-cli login

# Push LoRA adapters + head (lightweight, base model stays on Hub)
# best_model.push_to_hub(HUB_REPO_ID, token=HF_TOKEN)

# OR: merge LoRA into base and push full model
# best_model.merge_and_push(HUB_REPO_ID, token=HF_TOKEN)

print('Uncomment the line above and set your Hub credentials to push.')